In [1]:

"""
CN7 01 - 데이터 구조 진단
========================
목적
----
모델링/파생변수/shot 가설을 모두 배제하고 아래 6가지만 확인한다.

1) CN7 2차 가공 데이터가 1차 가공 데이터의 어디에서 왔는가?
2) 어떤 컬럼이 제거/유지되었는가?
3) 행 순서는 유지되었는가?
4) 숫자 컬럼은 어떤 방식으로 변환/스케일링되었는가?
5) PassOrFail 값이 실제로 무엇을 뜻하며 원본과 CN7에서 동일한가?
6) train/test 데이터 구조에 처음부터 눈에 띄는 이상이 있는가?

주의
----
- 이 스크립트는 0/1 중 어느 쪽이 불량인지 가정하지 않는다.
- 기존 pair/shot/regime/Mahalanobis 등의 가설을 사용하지 않는다.
- 모델 학습을 하지 않는다.
"""

from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 설정
# =============================================================================

# 기존 파일들이 들어 있는 폴더.
# 다른 PC/폴더에서 실행하면 여기만 수정하면 됩니다.
BASE_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋"
)

OUT_DIR = Path("cn7_01_diagnosis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 파일명은 대소문자를 무시하고 BASE_DIR 하위에서 재귀 탐색합니다.
FILE_CANDIDATES = {
    "labeled_data": [
        "labeled_data.csv",
        "labeled data.csv",
    ],
    "moldset_labeled": [
        "moldset_labeled.csv",
        "moldset labeled.csv",
    ],
    "supervised_cn7": [
        "supervised_label_cn7.csv",
        "supervised label cn7.csv",
    ],
    "moldset_labeled_cn7": [
        "moldset_labeled_cn7.csv",
        "moldset labeled cn7.csv",
    ],
    "unlabeled_data": [
        "unlabeled_data.csv",
        "unlabeled data.csv",
    ],
    "moldset_unlabeled_cn7": [
        "moldset_unlabeled_cn7.csv",
        "moldset unlabeled cn7.csv",
    ],
}

TARGET_CANDIDATES = ["PassOrFail", "passorfail", "PASSORFAIL"]

# 'CN7'이 포함된 원본 행을 찾을 때 우선 확인할 컬럼
PRODUCT_HINT_COLS = [
    "PART_NAME", "PART_NO", "PART_FACT_SERIAL",
    "EQUIP_NAME", "EQUIP_CD",
]

# 메타데이터/식별성 컬럼 후보
META_HINTS = {
    "id",
    "timestamp",
    "part_fact_plan_date",
    "part_fact_serial",
    "part_no",
    "part_name",
    "equip_cd",
    "equip_name",
    "reason",
    "err_fact_qty",
    "side",
}

# scaling/matching에서 사용하지 않을 인덱스성 컬럼
INDEX_LIKE = {
    "unnamed: 0",
    "index",
    "id",
}

# 최근접 매칭은 차원 수가 충분할 때만 수행
MIN_MATCH_FEATURES = 5

# exact-ish 판단은 아래 값보다 충분히 작을 때 참고
NN_EXACTISH_THRESHOLD = 1e-3

# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def normalize_name(x):
    return str(x).strip().lower()


def find_col(df, candidates):
    """대소문자 무시 컬럼 탐색."""
    m = {normalize_name(c): c for c in df.columns}
    for cand in candidates:
        if normalize_name(cand) in m:
            return m[normalize_name(cand)]
    return None


def find_file(base_dir: Path, candidates):
    """BASE_DIR 하위에서 후보 파일명을 대소문자 무시로 찾기."""
    if not base_dir.exists():
        return None

    candidate_lower = {c.lower() for c in candidates}

    # 정확한 파일명 우선
    for p in base_dir.rglob("*"):
        if p.is_file() and p.name.lower() in candidate_lower:
            return p

    return None


def read_csv_flexible(path):
    """한국어 CSV에서 흔한 인코딩까지 순차 시도."""
    encodings = [None, "utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_err = None

    for enc in encodings:
        try:
            if enc is None:
                return pd.read_csv(path)
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e

    raise RuntimeError(f"CSV 읽기 실패: {path}\n마지막 오류: {last_err}")


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def is_numeric_usable(df, c):
    if c not in df.columns:
        return False
    return pd.api.types.is_numeric_dtype(df[c])


def target_col(df):
    return find_col(df, TARGET_CANDIDATES)


def numeric_feature_columns(df, exclude_target=True):
    t = target_col(df)
    cols = []
    for c in df.columns:
        if exclude_target and t is not None and c == t:
            continue
        if normalize_name(c) in INDEX_LIKE:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols


def dataframe_overview(name, df):
    t = target_col(df)

    dup_all = int(df.duplicated().sum())

    rec = {
        "dataset": name,
        "rows": len(df),
        "cols": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_full_rows": dup_all,
        "target_col": t if t else "",
        "target_unique": int(df[t].nunique(dropna=False)) if t else np.nan,
    }

    if t:
        vc = df[t].value_counts(dropna=False).to_dict()
        rec["target_counts"] = json.dumps(
            {str(k): int(v) for k, v in vc.items()},
            ensure_ascii=False
        )
    else:
        rec["target_counts"] = ""

    return rec


def column_profile(name, df):
    rows = []
    for c in df.columns:
        s = df[c]
        row = {
            "dataset": name,
            "column": c,
            "dtype": str(s.dtype),
            "n_unique": int(s.nunique(dropna=False)),
            "missing": int(s.isna().sum()),
            "missing_rate": float(s.isna().mean()),
            "constant": bool(s.nunique(dropna=False) <= 1),
        }

        if pd.api.types.is_numeric_dtype(s):
            z = safe_numeric(s)
            row.update({
                "mean": float(z.mean()) if z.notna().any() else np.nan,
                "std_ddof1": float(z.std(ddof=1)) if z.notna().sum() > 1 else np.nan,
                "std_ddof0": float(z.std(ddof=0)) if z.notna().sum() > 0 else np.nan,
                "min": float(z.min()) if z.notna().any() else np.nan,
                "q01": float(z.quantile(.01)) if z.notna().any() else np.nan,
                "q25": float(z.quantile(.25)) if z.notna().any() else np.nan,
                "median": float(z.median()) if z.notna().any() else np.nan,
                "q75": float(z.quantile(.75)) if z.notna().any() else np.nan,
                "q99": float(z.quantile(.99)) if z.notna().any() else np.nan,
                "max": float(z.max()) if z.notna().any() else np.nan,
                "zero_rate": float((z == 0).mean()),
            })
        rows.append(row)

    return pd.DataFrame(rows)


def get_cn7_mask(df):
    """
    원본 데이터에서 CN7 텍스트가 들어간 행 탐색.
    특정 컬럼을 미리 단정하지 않고 모든 문자열/범주 컬럼도 확인한다.
    """
    masks = {}
    union = pd.Series(False, index=df.index)

    ordered_cols = []
    for h in PRODUCT_HINT_COLS:
        c = find_col(df, [h])
        if c and c not in ordered_cols:
            ordered_cols.append(c)

    for c in df.columns:
        if c not in ordered_cols:
            ordered_cols.append(c)

    for c in ordered_cols:
        s = df[c]
        # object/string/category뿐 아니라 코드가 문자열로 읽힐 수 있어 str 변환
        try:
            m = s.astype(str).str.contains("CN7", case=False, na=False, regex=False)
        except Exception:
            continue

        if int(m.sum()) > 0:
            masks[c] = m
            union = union | m

    return union, masks


def common_columns(a, b):
    b_map = {normalize_name(c): c for c in b.columns}
    pairs = []
    for ca in a.columns:
        key = normalize_name(ca)
        if key in b_map:
            pairs.append((ca, b_map[key]))
    return pairs


def compare_column_sets(name_a, a, name_b, b):
    a_map = {normalize_name(c): c for c in a.columns}
    b_map = {normalize_name(c): c for c in b.columns}

    only_a = [a_map[k] for k in a_map.keys() - b_map.keys()]
    only_b = [b_map[k] for k in b_map.keys() - a_map.keys()]
    shared = [(a_map[k], b_map[k]) for k in a_map.keys() & b_map.keys()]

    return {
        "a": name_a,
        "b": name_b,
        "n_a_cols": len(a.columns),
        "n_b_cols": len(b.columns),
        "n_shared": len(shared),
        "only_a": only_a,
        "only_b": only_b,
        "shared": [x[0] for x in shared],
    }


def standardize_with_pool(X, pool, cols, ddof=0):
    mu = pool[cols].mean()
    sd = pool[cols].std(ddof=ddof)
    sd = sd.mask(sd.abs() < 1e-12, np.nan)
    return (X[cols] - mu) / sd


def aligned_metrics(A, B):
    """
    동일한 행 수/순서를 가정하고 두 numeric matrix 차이 측정.
    """
    assert A.shape == B.shape
    diff = (A - B).abs()
    return {
        "mae": float(np.nanmean(diff.to_numpy())),
        "median_abs": float(np.nanmedian(diff.to_numpy())),
        "p95_abs": float(np.nanquantile(diff.to_numpy(), .95)),
        "max_abs": float(np.nanmax(diff.to_numpy())),
    }


def nearest_mapping(reference, query, ref_index, query_index):
    """
    query 각 행 -> reference에서 가장 가까운 행.
    NaN이 있는 행/컬럼은 사전에 제거된 상태를 기대.
    """
    nn = NearestNeighbors(n_neighbors=2, metric="euclidean", n_jobs=-1)
    nn.fit(reference)
    dist, ind = nn.kneighbors(query)

    out = pd.DataFrame({
        "query_row": query_index,
        "matched_source_row": np.asarray(ref_index)[ind[:, 0]],
        "nn_distance": dist[:, 0],
        "second_nn_distance": dist[:, 1] if dist.shape[1] > 1 else np.nan,
    })
    out["margin_to_2nd"] = out["second_nn_distance"] - out["nn_distance"]
    return out


def target_agreement(source, second, mapping=None):
    """
    원본 target과 2차 target의 값이 그대로 유지됐는지/반전됐는지 확인.
    mapping이 없으면 같은 길이일 때 행 순서대로 비교.
    """
    ts = target_col(source)
    tt = target_col(second)
    if ts is None or tt is None:
        return None

    if mapping is None:
        if len(source) != len(second):
            return None
        ys = source[ts].reset_index(drop=True)
        yt = second[tt].reset_index(drop=True)
    else:
        src_by_idx = source[ts]
        ys = mapping["matched_source_row"].map(src_by_idx).reset_index(drop=True)
        yt = second.loc[mapping["query_row"], tt].reset_index(drop=True)

    valid = ys.notna() & yt.notna()
    if valid.sum() == 0:
        return None

    ys = ys[valid]
    yt = yt[valid]

    same = float((ys.astype(str) == yt.astype(str)).mean())

    out = {
        "n_compared": int(valid.sum()),
        "same_value_rate": same,
        "source_values": sorted(pd.Series(ys).astype(str).unique().tolist()),
        "second_values": sorted(pd.Series(yt).astype(str).unique().tolist()),
    }

    # 둘 다 0/1인 경우 반전 일치율도 확인
    ys_num = pd.to_numeric(ys, errors="coerce")
    yt_num = pd.to_numeric(yt, errors="coerce")
    valid01 = ys_num.isin([0, 1]) & yt_num.isin([0, 1])

    if valid01.sum() > 0:
        out["binary_compared"] = int(valid01.sum())
        out["binary_same_rate"] = float(
            (ys_num[valid01].to_numpy() == yt_num[valid01].to_numpy()).mean()
        )
        out["binary_inverted_rate"] = float(
            ((1 - ys_num[valid01].to_numpy()) == yt_num[valid01].to_numpy()).mean()
        )

    return out


# =============================================================================
# 1. 파일 탐색 / 로드
# =============================================================================

banner("CN7 01 - 데이터 구조 진단")

print("BASE_DIR:", BASE_DIR)
print("OUT_DIR :", OUT_DIR.resolve())

paths = {}
datasets = {}

for key, names in FILE_CANDIDATES.items():
    p = find_file(BASE_DIR, names)
    paths[key] = p

    if p is None:
        print(f"[없음] {key:24s} | 후보={names}")
        continue

    try:
        df = read_csv_flexible(p)
        datasets[key] = df
        print(f"[로드] {key:24s} | {p.name:30s} | shape={df.shape}")
    except Exception as e:
        print(f"[실패] {key:24s} | {p} | {e}")

if not datasets:
    raise SystemExit(
        "\nCSV를 하나도 찾지 못했습니다.\n"
        "BASE_DIR 경로를 실제 데이터 폴더로 수정하세요."
    )

with open(OUT_DIR / "resolved_paths.json", "w", encoding="utf-8") as f:
    json.dump(
        {k: str(v) if v is not None else None for k, v in paths.items()},
        f,
        ensure_ascii=False,
        indent=2,
    )


# =============================================================================
# 2. 데이터셋 전체 구조
# =============================================================================

banner("1) 데이터셋 전체 구조")

overview = pd.DataFrame([
    dataframe_overview(name, df)
    for name, df in datasets.items()
])
print(overview.to_string(index=False))
overview.to_csv(OUT_DIR / "dataset_overview.csv", index=False, encoding="utf-8-sig")

profiles = []
for name, df in datasets.items():
    prof = column_profile(name, df)
    profiles.append(prof)
    prof.to_csv(
        OUT_DIR / f"column_profile__{name}.csv",
        index=False,
        encoding="utf-8-sig",
    )

all_profiles = pd.concat(profiles, ignore_index=True)
all_profiles.to_csv(
    OUT_DIR / "column_profiles_all.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 3. Target 값 - 절대 의미를 가정하지 않고 실제 값만 확인
# =============================================================================

banner("2) PassOrFail / Target 값 확인")

target_records = []

for name, df in datasets.items():
    t = target_col(df)
    if t is None:
        print(f"{name:24s}: target 없음")
        continue

    vc = df[t].value_counts(dropna=False)
    print(f"\n[{name}] target column = {t}")
    print(vc.to_string())

    for val, cnt in vc.items():
        target_records.append({
            "dataset": name,
            "target_col": t,
            "value": val,
            "count": int(cnt),
            "rate": float(cnt / len(df)),
        })

pd.DataFrame(target_records).to_csv(
    OUT_DIR / "target_value_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    "\n주의: 여기서는 0/1 중 어느 값이 불량인지 아직 결정하지 않습니다."
)
print(
    "      뒤에서 원본 labeled_data와 CN7 라벨의 동일/반전 여부를 직접 비교합니다."
)


# =============================================================================
# 4. 컬럼 관계: 1차 -> 2차에서 무엇이 남고 사라졌나
# =============================================================================

banner("3) 1차/2차 컬럼 관계")

pairs_to_compare = [
    ("labeled_data", "supervised_cn7"),
    ("labeled_data", "moldset_labeled_cn7"),
    ("moldset_labeled", "moldset_labeled_cn7"),
    ("unlabeled_data", "moldset_unlabeled_cn7"),
]

column_relation_rows = []

for a_name, b_name in pairs_to_compare:
    if a_name not in datasets or b_name not in datasets:
        continue

    r = compare_column_sets(
        a_name, datasets[a_name],
        b_name, datasets[b_name],
    )

    print(f"\n[{a_name} -> {b_name}]")
    print(f"  {a_name}: {r['n_a_cols']} cols")
    print(f"  {b_name}: {r['n_b_cols']} cols")
    print(f"  공통: {r['n_shared']} cols")
    print(f"  1차에만 존재: {r['only_a']}")
    print(f"  2차에만 존재: {r['only_b']}")

    column_relation_rows.append({
        "source": a_name,
        "second": b_name,
        "source_n_cols": r["n_a_cols"],
        "second_n_cols": r["n_b_cols"],
        "n_shared": r["n_shared"],
        "only_source": json.dumps(r["only_a"], ensure_ascii=False),
        "only_second": json.dumps(r["only_b"], ensure_ascii=False),
        "shared": json.dumps(r["shared"], ensure_ascii=False),
    })

pd.DataFrame(column_relation_rows).to_csv(
    OUT_DIR / "column_relationships.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 5. 원본에서 CN7 후보 행 탐색
# =============================================================================

banner("4) 원본에서 CN7 후보 행 탐색")

source_cn7_candidates = {}

for source_name in ["labeled_data", "moldset_labeled", "unlabeled_data"]:
    if source_name not in datasets:
        continue

    df = datasets[source_name]
    mask, masks_by_col = get_cn7_mask(df)

    print(f"\n[{source_name}] CN7 텍스트 포함 행: {int(mask.sum())} / {len(df)}")

    for c, m in masks_by_col.items():
        vals = (
            df.loc[m, c]
            .astype(str)
            .value_counts()
            .head(20)
        )
        print(f"  - {c}: {int(m.sum())} rows")
        print("    값 예시:")
        for v, cnt in vals.items():
            print(f"      {v!r}: {cnt}")

    if int(mask.sum()) > 0:
        source_cn7_candidates[source_name] = df.loc[mask].copy()

        # 원본 index 보존
        out = df.loc[mask].copy()
        out.insert(0, "__source_row_index__", out.index)
        out.to_csv(
            OUT_DIR / f"cn7_candidate_rows__{source_name}.csv",
            index=False,
            encoding="utf-8-sig",
        )


# =============================================================================
# 6. CN7 2차 데이터 자체가 표준화됐는지
# =============================================================================

banner("5) CN7 2차 데이터 스케일링 상태")

scale_rows = []

for name in ["supervised_cn7", "moldset_labeled_cn7", "moldset_unlabeled_cn7"]:
    if name not in datasets:
        continue

    df = datasets[name]
    t = target_col(df)

    num_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
        and c != t
        and normalize_name(c) not in INDEX_LIKE
    ]

    print(f"\n[{name}] numeric features = {len(num_cols)}")

    for c in num_cols:
        s = safe_numeric(df[c])
        mean = s.mean()
        std0 = s.std(ddof=0)
        std1 = s.std(ddof=1)

        scale_rows.append({
            "dataset": name,
            "column": c,
            "mean": mean,
            "std_ddof0": std0,
            "std_ddof1": std1,
            "near_zero_mean": bool(abs(mean) < 0.05) if pd.notna(mean) else False,
            "near_unit_std": bool(abs(std0 - 1) < 0.05) if pd.notna(std0) else False,
            "n_unique": int(s.nunique(dropna=False)),
            "min": s.min(),
            "max": s.max(),
        })

    local = pd.DataFrame([r for r in scale_rows if r["dataset"] == name])

    if len(local):
        print(
            "  평균≈0 비율:",
            f"{local['near_zero_mean'].mean():.1%}",
            "| std≈1 비율:",
            f"{local['near_unit_std'].mean():.1%}",
        )

        print(
            local[
                ["column", "mean", "std_ddof0", "n_unique", "min", "max"]
            ]
            .round(4)
            .to_string(index=False)
        )

scale_df = pd.DataFrame(scale_rows)
scale_df.to_csv(
    OUT_DIR / "second_stage_scaling_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 7. 1차 CN7 후보 -> 2차 CN7 변환 관계 테스트
# =============================================================================

banner("6) 1차 CN7 후보 ↔ 2차 CN7 변환/행순서 테스트")

comparison_jobs = []

# 지도 CN7은 labeled_data/moldset_labeled 양쪽 후보와 비교
for src in ["labeled_data", "moldset_labeled"]:
    for sec in ["supervised_cn7", "moldset_labeled_cn7"]:
        if src in source_cn7_candidates and sec in datasets:
            comparison_jobs.append((src, sec))

# 비지도 CN7은 unlabeled_data와 비교
if "unlabeled_data" in source_cn7_candidates and "moldset_unlabeled_cn7" in datasets:
    comparison_jobs.append(("unlabeled_data", "moldset_unlabeled_cn7"))

transform_summary = []
mapping_files = []

for src_name, sec_name in comparison_jobs:
    src_all = datasets[src_name]
    src = source_cn7_candidates[src_name].copy()
    sec = datasets[sec_name].copy()

    t_src = target_col(src)
    t_sec = target_col(sec)

    # 이름이 같은 numeric 컬럼
    sec_map = {normalize_name(c): c for c in sec.columns}

    common_num = []
    for c_src in src.columns:
        key = normalize_name(c_src)

        if key not in sec_map:
            continue

        c_sec = sec_map[key]

        if c_src == t_src or c_sec == t_sec:
            continue

        if normalize_name(c_src) in INDEX_LIKE:
            continue

        if (
            pd.api.types.is_numeric_dtype(src[c_src])
            and pd.api.types.is_numeric_dtype(sec[c_sec])
        ):
            common_num.append((c_src, c_sec))

    print(f"\n[{src_name} -> {sec_name}]")
    print(f"  source CN7 후보: {len(src)} rows")
    print(f"  second-stage   : {len(sec)} rows")
    print(f"  공통 numeric   : {len(common_num)} cols")

    if len(common_num) < MIN_MATCH_FEATURES:
        print("  -> 공통 numeric 컬럼 부족. 변환/매칭 생략.")
        continue

    src_cols = [x[0] for x in common_num]
    sec_cols = [x[1] for x in common_num]

    # 내부 계산 편의를 위해 second 컬럼명을 source 이름으로 맞춤
    sec_num = sec[sec_cols].copy()
    sec_num.columns = src_cols

    # 변환 방법 세 가지
    transforms = {}

    # A) raw 그대로
    transforms["raw"] = src[src_cols].astype(float)

    # B) CN7 후보 자체 평균/std(ddof=0)로 z-score
    transforms["z_by_cn7_candidate"] = standardize_with_pool(
        src, src, src_cols, ddof=0
    )

    # C) 1차 전체 데이터 평균/std(ddof=0)로 z-score
    full_usable = src_all[src_cols].apply(pd.to_numeric, errors="coerce")
    src_usable = src[src_cols].apply(pd.to_numeric, errors="coerce")

    mu_full = full_usable.mean()
    sd_full = full_usable.std(ddof=0).replace(0, np.nan)
    transforms["z_by_full_source"] = (src_usable - mu_full) / sd_full

    # D) sample std(ddof=1) 가능성도 참고
    transforms["z_by_cn7_candidate_ddof1"] = standardize_with_pool(
        src, src, src_cols, ddof=1
    )

    # 비교에 쓸 컬럼: 양쪽 모두 finite한 열
    sec_mat0 = sec_num.replace([np.inf, -np.inf], np.nan)

    for method, ref_df in transforms.items():
        ref_df = ref_df.replace([np.inf, -np.inf], np.nan)

        valid_cols = [
            c for c in src_cols
            if ref_df[c].notna().all() and sec_mat0[c].notna().all()
        ]

        if len(valid_cols) < MIN_MATCH_FEATURES:
            continue

        ref_mat = ref_df[valid_cols].to_numpy(float)
        q_mat = sec_mat0[valid_cols].to_numpy(float)

        row = {
            "source": src_name,
            "second": sec_name,
            "method": method,
            "source_rows": len(src),
            "second_rows": len(sec),
            "n_features": len(valid_cols),
        }

        # 행 순서 그대로 비교 가능할 때
        if len(src) == len(sec):
            m = aligned_metrics(
                pd.DataFrame(ref_mat),
                pd.DataFrame(q_mat),
            )
            row.update({f"aligned_{k}": v for k, v in m.items()})
        else:
            row.update({
                "aligned_mae": np.nan,
                "aligned_median_abs": np.nan,
                "aligned_p95_abs": np.nan,
                "aligned_max_abs": np.nan,
            })

        # NN 매칭
        mapping = nearest_mapping(
            reference=ref_mat,
            query=q_mat,
            ref_index=src.index.to_numpy(),
            query_index=sec.index.to_numpy(),
        )

        row["nn_median_dist"] = float(mapping["nn_distance"].median())
        row["nn_p95_dist"] = float(mapping["nn_distance"].quantile(.95))
        row["nn_max_dist"] = float(mapping["nn_distance"].max())
        row["unique_source_match_rate"] = float(
            mapping["matched_source_row"].nunique() / len(mapping)
        )
        row["exactish_rate"] = float(
            (mapping["nn_distance"] <= NN_EXACTISH_THRESHOLD).mean()
        )

        transform_summary.append(row)

        map_path = OUT_DIR / (
            f"mapping__{src_name}__to__{sec_name}__{method}.csv"
        )
        mapping.to_csv(map_path, index=False, encoding="utf-8-sig")
        mapping_files.append(map_path)

        print(
            f"  {method:28s} | "
            f"aligned MAE={row['aligned_mae']:.6g} | "
            f"NN median={row['nn_median_dist']:.6g} | "
            f"NN p95={row['nn_p95_dist']:.6g} | "
            f"unique={row['unique_source_match_rate']:.3f} | "
            f"exactish={row['exactish_rate']:.3f}"
        )

transform_df = pd.DataFrame(transform_summary)

if len(transform_df):
    transform_df = transform_df.sort_values(
        ["source", "second", "nn_median_dist", "nn_p95_dist"]
    )

    transform_df.to_csv(
        OUT_DIR / "source_to_second_transform_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =============================================================================
# 8. 각 source-second 조합에서 가장 가까운 변환으로 target 유지/반전 검사
# =============================================================================

banner("7) Target 유지/반전 여부")

target_relation_records = []

if len(transform_df):
    for (src_name, sec_name), part in transform_df.groupby(["source", "second"]):
        best = part.sort_values(
            ["nn_median_dist", "nn_p95_dist"]
        ).iloc[0]

        method = best["method"]

        mapping_path = OUT_DIR / (
            f"mapping__{src_name}__to__{sec_name}__{method}.csv"
        )

        if not mapping_path.exists():
            continue

        mapping = pd.read_csv(mapping_path)
        src = source_cn7_candidates[src_name]
        sec = datasets[sec_name]

        result = target_agreement(src, sec, mapping)

        print(f"\n[{src_name} -> {sec_name}] best method = {method}")
        print(
            f"  NN median={best['nn_median_dist']:.6g}, "
            f"p95={best['nn_p95_dist']:.6g}, "
            f"unique={best['unique_source_match_rate']:.3f}"
        )

        if result is None:
            print("  Target 비교 불가")
            continue

        for k, v in result.items():
            print(f"  {k}: {v}")

        target_relation_records.append({
            "source": src_name,
            "second": sec_name,
            "best_transform": method,
            **result,
        })

target_relation_df = pd.DataFrame(target_relation_records)
target_relation_df.to_csv(
    OUT_DIR / "target_relationships.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 9. 2차 지도 데이터끼리 직접 비교
# =============================================================================

banner("8) supervised_label_cn7 vs moldset_labeled_cn7")

if "supervised_cn7" in datasets and "moldset_labeled_cn7" in datasets:
    a = datasets["supervised_cn7"]
    b = datasets["moldset_labeled_cn7"]

    print("supervised_cn7      :", a.shape)
    print("moldset_labeled_cn7:", b.shape)

    common = common_columns(a, b)
    print("공통 컬럼:", len(common))

    if len(a) == len(b):
        equal_cols = []
        diff_cols = []

        for ca, cb in common:
            sa, sb = a[ca], b[cb]

            if (
                pd.api.types.is_numeric_dtype(sa)
                and pd.api.types.is_numeric_dtype(sb)
            ):
                x = pd.to_numeric(sa, errors="coerce")
                y = pd.to_numeric(sb, errors="coerce")
                same = bool(
                    np.allclose(
                        x.to_numpy(float),
                        y.to_numpy(float),
                        rtol=1e-7,
                        atol=1e-9,
                        equal_nan=True,
                    )
                )
            else:
                same = bool(
                    sa.fillna("__NA__").astype(str).reset_index(drop=True)
                    .equals(
                        sb.fillna("__NA__").astype(str).reset_index(drop=True)
                    )
                )

            (equal_cols if same else diff_cols).append(ca)

        print("행 순서 그대로 완전 동일 컬럼:", equal_cols)
        print("값이 다른 공통 컬럼:", diff_cols)

        pd.DataFrame({
            "equal_columns": pd.Series(equal_cols),
            "different_columns": pd.Series(diff_cols),
        }).to_csv(
            OUT_DIR / "supervised_vs_moldset_cn7_columns.csv",
            index=False,
            encoding="utf-8-sig",
        )
    else:
        print("행 수가 달라 직접 행 순서 비교는 하지 않음.")
else:
    print("두 파일 중 하나 이상이 없어 비교하지 못했습니다.")


# =============================================================================
# 10. 중복/인접행 구조 - 관찰만 하고 shot이라고 해석하지 않음
# =============================================================================

banner("9) 중복 / 인접행 구조 진단")

duplicate_records = []

for name in ["supervised_cn7", "moldset_labeled_cn7", "moldset_unlabeled_cn7"]:
    if name not in datasets:
        continue

    df = datasets[name]
    t = target_col(df)

    feat_cols = [
        c for c in df.columns
        if c != t
        and normalize_name(c) not in INDEX_LIKE
    ]

    numeric_feats = [
        c for c in feat_cols
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    dup_feature = df.duplicated(subset=feat_cols, keep=False)
    duplicated_rows = int(dup_feature.sum())
    duplicated_groups = (
        int(df.loc[dup_feature].groupby(feat_cols, dropna=False).ngroups)
        if duplicated_rows > 0 else 0
    )

    rec = {
        "dataset": name,
        "rows": len(df),
        "feature_duplicate_rows": duplicated_rows,
        "feature_duplicate_rate": duplicated_rows / len(df) if len(df) else np.nan,
        "feature_duplicate_groups": duplicated_groups,
    }

    # 같은 X인데 target이 다른 그룹
    if t is not None and duplicated_rows > 0:
        g = (
            df.groupby(feat_cols, dropna=False)[t]
            .nunique(dropna=False)
        )
        conflict_groups = int((g > 1).sum())
        rec["same_X_different_target_groups"] = conflict_groups
    else:
        rec["same_X_different_target_groups"] = np.nan

    # 인접 numeric 행이 완전히/거의 같은 비율
    if len(numeric_feats) >= 1 and len(df) >= 2:
        X = df[numeric_feats].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        diff = np.nanmax(np.abs(X[1:] - X[:-1]), axis=1)

        rec["adjacent_exact_rate_1e-12"] = float(np.nanmean(diff <= 1e-12))
        rec["adjacent_near_rate_1e-6"] = float(np.nanmean(diff <= 1e-6))
        rec["adjacent_diff_median"] = float(np.nanmedian(diff))
        rec["adjacent_diff_p95"] = float(np.nanquantile(diff, .95))
    else:
        rec["adjacent_exact_rate_1e-12"] = np.nan
        rec["adjacent_near_rate_1e-6"] = np.nan
        rec["adjacent_diff_median"] = np.nan
        rec["adjacent_diff_p95"] = np.nan

    duplicate_records.append(rec)

    print(f"\n[{name}]")
    for k, v in rec.items():
        if k != "dataset":
            print(f"  {k}: {v}")

duplicate_df = pd.DataFrame(duplicate_records)
duplicate_df.to_csv(
    OUT_DIR / "duplicate_and_adjacency_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 11. train/test 2차 데이터 구조 차이 - 아직 모델링 X
# =============================================================================

banner("10) moldset CN7 train/test 기초 비교")

if "moldset_labeled_cn7" in datasets and "moldset_unlabeled_cn7" in datasets:
    tr = datasets["moldset_labeled_cn7"]
    te = datasets["moldset_unlabeled_cn7"]

    t = target_col(tr)

    te_map = {normalize_name(c): c for c in te.columns}

    common_num = []
    for c in tr.columns:
        if c == t or normalize_name(c) in INDEX_LIKE:
            continue

        key = normalize_name(c)
        if key not in te_map:
            continue

        ct = te_map[key]

        if (
            pd.api.types.is_numeric_dtype(tr[c])
            and pd.api.types.is_numeric_dtype(te[ct])
        ):
            common_num.append((c, ct))

    train_test_rows = []

    for c_tr, c_te in common_num:
        a = pd.to_numeric(tr[c_tr], errors="coerce")
        b = pd.to_numeric(te[c_te], errors="coerce")

        row = {
            "column": c_tr,
            "train_mean": a.mean(),
            "train_std0": a.std(ddof=0),
            "train_min": a.min(),
            "train_q25": a.quantile(.25),
            "train_median": a.median(),
            "train_q75": a.quantile(.75),
            "train_max": a.max(),
            "train_unique": a.nunique(dropna=False),

            "test_mean": b.mean(),
            "test_std0": b.std(ddof=0),
            "test_min": b.min(),
            "test_q25": b.quantile(.25),
            "test_median": b.median(),
            "test_q75": b.quantile(.75),
            "test_max": b.max(),
            "test_unique": b.nunique(dropna=False),
        }

        row["mean_abs_diff"] = abs(row["train_mean"] - row["test_mean"])
        row["std_abs_diff"] = abs(row["train_std0"] - row["test_std0"])

        train_test_rows.append(row)

    train_test_df = pd.DataFrame(train_test_rows)

    if len(train_test_df):
        train_test_df = train_test_df.sort_values(
            ["mean_abs_diff", "std_abs_diff"],
            ascending=False,
        )

        print(
            train_test_df[
                [
                    "column",
                    "train_mean", "train_std0", "train_unique",
                    "test_mean", "test_std0", "test_unique",
                    "mean_abs_diff", "std_abs_diff",
                ]
            ]
            .round(4)
            .to_string(index=False)
        )

        train_test_df.to_csv(
            OUT_DIR / "cn7_train_test_basic_distribution.csv",
            index=False,
            encoding="utf-8-sig",
        )
else:
    print("moldset_labeled_cn7 / moldset_unlabeled_cn7 둘 중 하나 이상이 없습니다.")


# =============================================================================
# 12. 자동 결론 요약
# =============================================================================

banner("11) 자동 진단 요약")

summary_lines = []

# Target
if len(target_relation_df):
    for _, r in target_relation_df.iterrows():
        msg = (
            f"- Target {r['source']} -> {r['second']}: "
            f"same={r.get('binary_same_rate', np.nan):.3f}, "
            f"inverted={r.get('binary_inverted_rate', np.nan):.3f}"
        )
        summary_lines.append(msg)

# Transform
if len(transform_df):
    for (src, sec), part in transform_df.groupby(["source", "second"]):
        best = part.sort_values(["nn_median_dist", "nn_p95_dist"]).iloc[0]
        msg = (
            f"- {src} -> {sec} 최적 후보 변환: {best['method']} | "
            f"NN median={best['nn_median_dist']:.6g}, "
            f"p95={best['nn_p95_dist']:.6g}, "
            f"unique={best['unique_source_match_rate']:.3f}, "
            f"exactish={best['exactish_rate']:.3f}"
        )
        summary_lines.append(msg)

# Scaling
if len(scale_df):
    for name, p in scale_df.groupby("dataset"):
        summary_lines.append(
            f"- {name}: mean≈0 컬럼 {p['near_zero_mean'].mean():.1%}, "
            f"std≈1 컬럼 {p['near_unit_std'].mean():.1%}"
        )

# Duplicates
if len(duplicate_df):
    for _, r in duplicate_df.iterrows():
        summary_lines.append(
            f"- {r['dataset']}: feature 중복행 비율="
            f"{r['feature_duplicate_rate']:.1%}, "
            f"same-X/different-target groups="
            f"{r['same_X_different_target_groups']}"
        )

if not summary_lines:
    summary_lines.append("- 자동 요약할 결과가 충분하지 않습니다. 누락 파일을 확인하세요.")

for line in summary_lines:
    print(line)

with open(
    OUT_DIR / "diagnosis_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(summary_lines))


# =============================================================================
# 13. 다음 단계에서 보내줄 핵심 파일/출력 안내
# =============================================================================

banner("완료")

print(f"결과 폴더: {OUT_DIR.resolve()}")
print("\n우선 확인할 파일:")
print("  1) dataset_overview.csv")
print("  2) target_value_counts.csv")
print("  3) column_relationships.csv")
print("  4) source_to_second_transform_summary.csv  (생성된 경우)")
print("  5) target_relationships.csv                (생성된 경우)")
print("  6) second_stage_scaling_summary.csv")
print("  7) duplicate_and_adjacency_summary.csv")
print("  8) diagnosis_summary.txt")

print(
    "\n모델링은 아직 하지 않습니다. "
    "위 결과로 '데이터가 어떻게 만들어졌는지'를 확정한 뒤 02단계로 넘어갑니다."
)



CN7 01 - 데이터 구조 진단
BASE_DIR: C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋
OUT_DIR : C:\Users\Administrator\Desktop\kamp\cn7\cn7_01_diagnosis
[로드] labeled_data             | labeled_data.csv               | shape=(7996, 45)
[로드] moldset_labeled          | moldset_labeled.csv            | shape=(2607, 47)
[로드] supervised_cn7           | supervised_label_cn7.csv       | shape=(6736, 26)
[로드] moldset_labeled_cn7      | moldset_labeled_cn7.csv        | shape=(1211, 26)
[로드] unlabeled_data           | unlabeled_data.csv             | shape=(795315, 46)
[로드] moldset_unlabeled_cn7    | moldset_unlabeled_cn7.csv      | shape=(35239, 25)

1) 데이터셋 전체 구조
              dataset   rows  cols  missing_cells  duplicate_full_rows target_col  target_unique        target_counts
         labeled_data   7996    45           7925                 2764 PassOrFail            2.0 {"Y": 7925, "N": 71}
      moldset_labeled   2607    47           2555                    0 PassOrFail      

In [2]:

"""
CN7 02 - 1425 -> 1211 필터 규칙 탐색 + 원본행 복원
=====================================================

목표
----
1) moldset_labeled.csv 에서 CN7 1425행 추출
2) moldset_labeled_cn7.csv 의 1211행이 어떤 원본 subset인지 탐색
3) StandardScaler(ddof=0)로 재현 가능한지 검증
4) 원본 metadata(TimeStamp, PART_NAME, PART_FACT_SERIAL 등) 복원
5) RH/LH Side 추출
6) PassOrFail 0/1이 원본과 동일/반전인지 검증

중요
----
- 기존 shot/pair 가설을 사용하지 않는다.
- 1=불량이라고 가정하지 않는다.
- 먼저 "어떤 1211개 행인지"를 데이터로 복원한다.
"""

from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 경로
# =============================================================================

BASE_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋"
)

OUT_DIR = Path("cn7_02_restore")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NAMES = [
    "moldset_labeled.csv",
    "moldset labeled.csv",
]

TARGET_NAMES = [
    "moldset_labeled_cn7.csv",
    "moldset labeled cn7.csv",
]

TARGET_CANDIDATES = ["PassOrFail", "passorfail", "PASSORFAIL"]

INDEX_LIKE = {"unnamed: 0", "index", "_id", "id"}

# Hungarian assignment에서 label 불일치를 얼마나 강하게 막을지
LABEL_MISMATCH_PENALTY = 1_000.0

# 재현이 사실상 완벽하다고 볼 거리 기준
EXACT_DISTANCE = 1e-4

# iterative assignment 최대 반복
MAX_ITER = 20


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 105)
    print(title)
    print("=" * 105)


def norm(x):
    return str(x).strip().lower()


def find_file(base_dir, names):
    names = {x.lower() for x in names}
    for p in base_dir.rglob("*"):
        if p.is_file() and p.name.lower() in names:
            return p
    return None


def read_csv_flexible(path):
    last = None
    for enc in [None, "utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            if enc is None:
                return pd.read_csv(path)
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise RuntimeError(f"CSV 읽기 실패: {path}\n{last}")


def find_col(df, candidates):
    m = {norm(c): c for c in df.columns}
    for x in candidates:
        if norm(x) in m:
            return m[norm(x)]
    return None


def target_col(df):
    return find_col(df, TARGET_CANDIDATES)


def get_cn7_mask(df):
    # 가장 직접적인 PART_NAME 우선
    part_name = find_col(df, ["PART_NAME"])
    if part_name is not None:
        return df[part_name].astype(str).str.contains(
            "CN7", case=False, na=False, regex=False
        )

    # fallback: 모든 컬럼
    mask = pd.Series(False, index=df.index)
    for c in df.columns:
        try:
            mask |= df[c].astype(str).str.contains(
                "CN7", case=False, na=False, regex=False
            )
        except Exception:
            pass
    return mask


def extract_side(series):
    s = series.astype(str).str.upper()
    out = pd.Series("UNKNOWN", index=series.index, dtype="object")
    out[s.str.contains(r"\bRH\b", regex=True, na=False)] = "RH"
    out[s.str.contains(r"\bLH\b", regex=True, na=False)] = "LH"
    return out


def common_numeric_features(source, target):
    t_src = target_col(source)
    t_tgt = target_col(target)

    target_map = {norm(c): c for c in target.columns}

    pairs = []
    for cs in source.columns:
        key = norm(cs)

        if key in INDEX_LIKE:
            continue
        if cs == t_src:
            continue
        if key not in target_map:
            continue

        ct = target_map[key]

        if ct == t_tgt:
            continue

        if (
            pd.api.types.is_numeric_dtype(source[cs])
            and pd.api.types.is_numeric_dtype(target[ct])
        ):
            pairs.append((cs, ct))

    return pairs


def scale_subset(subset, feature_cols):
    """
    sklearn StandardScaler와 같은 ddof=0.
    상수 컬럼은 0으로 출력.
    """
    X = subset[feature_cols].astype(float)
    mu = X.mean()
    sd = X.std(ddof=0)

    Z = pd.DataFrame(index=X.index, columns=feature_cols, dtype=float)

    for c in feature_cols:
        if pd.isna(sd[c]) or abs(sd[c]) < 1e-12:
            Z[c] = 0.0
        else:
            Z[c] = (X[c] - mu[c]) / sd[c]

    return Z, mu, sd


def direct_error(Z, T):
    A = Z.to_numpy(float)
    B = T.to_numpy(float)

    if A.shape != B.shape:
        return None

    d = np.sqrt(np.nansum((A - B) ** 2, axis=1))

    return {
        "mean_dist": float(np.mean(d)),
        "median_dist": float(np.median(d)),
        "p95_dist": float(np.quantile(d, .95)),
        "max_dist": float(np.max(d)),
        "exact_rate": float((d <= EXACT_DISTANCE).mean()),
    }


def prepare_binary_target(s):
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().all() and set(x.unique()).issubset({0, 1}):
        return x.astype(int)
    return None


def assignment_cost(
    Z_source,
    T_target,
    source_y=None,
    target_y=None,
    label_mode="none",
):
    """
    rectangular Hungarian:
    target 1211 rows -> source candidate rows

    label_mode:
      none      : 센서만
      same      : source target == target target 선호
      inverted  : source target == 1-target 선호
    """
    A = T_target.to_numpy(float)
    B = Z_source.to_numpy(float)

    C = cdist(A, B, metric="euclidean")

    if (
        label_mode != "none"
        and source_y is not None
        and target_y is not None
    ):
        sy = np.asarray(source_y)
        ty = np.asarray(target_y)

        if label_mode == "same":
            expected = ty
        elif label_mode == "inverted":
            expected = 1 - ty
        else:
            expected = None

        if expected is not None:
            mismatch = expected[:, None] != sy[None, :]
            C = C + mismatch.astype(float) * LABEL_MISMATCH_PENALTY

    return C


def hungarian_map(
    source_df,
    target_df,
    feature_cols,
    target_feature_cols,
    label_mode="none",
):
    Z, mu, sd = scale_subset(source_df, feature_cols)

    Z2 = Z.copy()
    Z2.columns = target_feature_cols

    T = target_df[target_feature_cols].astype(float)

    ys_col = target_col(source_df)
    yt_col = target_col(target_df)

    sy = prepare_binary_target(source_df[ys_col]) if ys_col else None
    ty = prepare_binary_target(target_df[yt_col]) if yt_col else None

    C = assignment_cost(
        Z2,
        T,
        source_y=sy.to_numpy() if sy is not None else None,
        target_y=ty.to_numpy() if ty is not None else None,
        label_mode=label_mode,
    )

    row_ind, col_ind = linear_sum_assignment(C)

    # row_ind = target row position
    # col_ind = source row position
    raw_dist = cdist(
        T.to_numpy(float),
        Z2.to_numpy(float),
        metric="euclidean",
    )[row_ind, col_ind]

    mapping = pd.DataFrame({
        "target_pos": row_ind,
        "source_pos": col_ind,
        "target_row_index": target_df.index.to_numpy()[row_ind],
        "source_row_index": source_df.index.to_numpy()[col_ind],
        "sensor_distance": raw_dist,
    })

    if sy is not None and ty is not None:
        mapped_sy = sy.to_numpy()[col_ind]
        mapped_ty = ty.to_numpy()[row_ind]

        mapping["source_target"] = mapped_sy
        mapping["target_target"] = mapped_ty
        mapping["target_same"] = mapped_sy == mapped_ty
        mapping["target_inverted"] = mapped_sy == (1 - mapped_ty)

    summary = {
        "label_mode": label_mode,
        "n": len(mapping),
        "median_dist": float(mapping["sensor_distance"].median()),
        "mean_dist": float(mapping["sensor_distance"].mean()),
        "p95_dist": float(mapping["sensor_distance"].quantile(.95)),
        "max_dist": float(mapping["sensor_distance"].max()),
        "exact_rate": float((mapping["sensor_distance"] <= EXACT_DISTANCE).mean()),
    }

    if "target_same" in mapping.columns:
        summary["target_same_rate"] = float(mapping["target_same"].mean())
        summary["target_inverted_rate"] = float(mapping["target_inverted"].mean())

    return mapping, summary, mu, sd


def score_exact_subset(
    subset,
    target,
    source_feature_cols,
    target_feature_cols,
    subset_name,
):
    """
    정확히 1211행짜리 후보 subset 평가.
    direct order + Hungarian(same/inverted/none) 모두 확인.
    """
    records = []
    outputs = {}

    Z, mu, sd = scale_subset(subset, source_feature_cols)
    Z_cmp = Z.copy()
    Z_cmp.columns = target_feature_cols

    direct = direct_error(
        Z_cmp.reset_index(drop=True),
        target[target_feature_cols].astype(float).reset_index(drop=True),
    )

    if direct is not None:
        records.append({
            "subset": subset_name,
            "mapping": "direct_order",
            **direct,
        })

    for mode in ["none", "same", "inverted"]:
        mapping, summary, mu2, sd2 = hungarian_map(
            subset,
            target,
            source_feature_cols,
            target_feature_cols,
            label_mode=mode,
        )

        rec = {
            "subset": subset_name,
            "mapping": f"hungarian_{mode}",
            "mean_dist": summary["mean_dist"],
            "median_dist": summary["median_dist"],
            "p95_dist": summary["p95_dist"],
            "max_dist": summary["max_dist"],
            "exact_rate": summary["exact_rate"],
            "target_same_rate": summary.get("target_same_rate", np.nan),
            "target_inverted_rate": summary.get("target_inverted_rate", np.nan),
        }
        records.append(rec)
        outputs[f"hungarian_{mode}"] = (mapping, mu2, sd2)

    return pd.DataFrame(records), outputs


def iterative_assignment(
    source_all,
    target,
    source_feature_cols,
    target_feature_cols,
    label_mode="none",
    max_iter=20,
):
    """
    exact filter가 안 보일 때 사용하는 fallback.

    1) 전체 1425 기준 scaler
    2) target 1211 -> source 1425 Hungarian assignment
    3) 선택된 1211만으로 scaler 재계산
    4) 반복
    """
    selected_idx = source_all.index.to_numpy()
    current_subset = source_all.copy()

    history = []
    prev_selected = None
    final_mapping = None
    final_mu = None
    final_sd = None

    for it in range(1, max_iter + 1):
        # 현재 선택 subset으로 scaler 파라미터 추정
        _, mu, sd = scale_subset(
            current_subset,
            source_feature_cols,
        )

        # 그 scaler로 "전체 1425"를 변환
        X_all = source_all[source_feature_cols].astype(float)
        Z_all = pd.DataFrame(
            index=source_all.index,
            columns=source_feature_cols,
            dtype=float,
        )

        for c in source_feature_cols:
            if pd.isna(sd[c]) or abs(sd[c]) < 1e-12:
                Z_all[c] = 0.0
            else:
                Z_all[c] = (X_all[c] - mu[c]) / sd[c]

        Z_all.columns = target_feature_cols
        T = target[target_feature_cols].astype(float)

        ys_col = target_col(source_all)
        yt_col = target_col(target)

        sy = prepare_binary_target(source_all[ys_col]) if ys_col else None
        ty = prepare_binary_target(target[yt_col]) if yt_col else None

        C = assignment_cost(
            Z_all,
            T,
            source_y=sy.to_numpy() if sy is not None else None,
            target_y=ty.to_numpy() if ty is not None else None,
            label_mode=label_mode,
        )

        row_ind, col_ind = linear_sum_assignment(C)

        raw_C = cdist(
            T.to_numpy(float),
            Z_all.to_numpy(float),
            metric="euclidean",
        )
        d = raw_C[row_ind, col_ind]

        selected_positions = np.sort(col_ind)
        selected_source_idx = source_all.index.to_numpy()[selected_positions]

        hist = {
            "iter": it,
            "label_mode": label_mode,
            "median_dist": float(np.median(d)),
            "mean_dist": float(np.mean(d)),
            "p95_dist": float(np.quantile(d, .95)),
            "max_dist": float(np.max(d)),
            "exact_rate": float((d <= EXACT_DISTANCE).mean()),
            "selected_changed": (
                np.nan if prev_selected is None
                else int(
                    len(
                        set(selected_source_idx)
                        .symmetric_difference(set(prev_selected))
                    )
                )
            ),
        }
        history.append(hist)

        mapping = pd.DataFrame({
            "target_pos": row_ind,
            "source_pos": col_ind,
            "target_row_index": target.index.to_numpy()[row_ind],
            "source_row_index": source_all.index.to_numpy()[col_ind],
            "sensor_distance": d,
        })

        if sy is not None and ty is not None:
            mapped_sy = sy.to_numpy()[col_ind]
            mapped_ty = ty.to_numpy()[row_ind]
            mapping["source_target"] = mapped_sy
            mapping["target_target"] = mapped_ty
            mapping["target_same"] = mapped_sy == mapped_ty
            mapping["target_inverted"] = mapped_sy == (1 - mapped_ty)

        final_mapping = mapping
        final_mu = mu
        final_sd = sd

        if prev_selected is not None and set(selected_source_idx) == set(prev_selected):
            break

        prev_selected = selected_source_idx
        current_subset = source_all.loc[selected_source_idx].copy()

    return (
        pd.DataFrame(history),
        final_mapping,
        current_subset,
        final_mu,
        final_sd,
    )


def affine_column_check(
    subset,
    target,
    mapping,
    source_feature_cols,
    target_feature_cols,
):
    """
    매핑된 raw x와 target z에 대해 x = intercept + slope*z를 fitting.
    StandardScaler가 맞으면:
      intercept ~= subset mean
      slope ~= subset std(ddof=0)
      residual ~= 0
    """
    rows = []

    map2 = mapping.sort_values("target_pos")

    for cs, ct in zip(source_feature_cols, target_feature_cols):
        x = subset.loc[map2["source_row_index"], cs].to_numpy(float)
        z = target.loc[map2["target_row_index"], ct].to_numpy(float)

        if np.std(z) < 1e-12:
            slope = np.nan
            intercept = float(np.mean(x))
            pred = np.repeat(intercept, len(x))
        else:
            slope, intercept = np.polyfit(z, x, deg=1)
            pred = intercept + slope * z

        resid = x - pred

        subset_mean = subset[cs].astype(float).mean()
        subset_std0 = subset[cs].astype(float).std(ddof=0)

        rows.append({
            "feature": cs,
            "affine_intercept": intercept,
            "subset_mean": subset_mean,
            "intercept_minus_mean": intercept - subset_mean,
            "affine_slope": slope,
            "subset_std0": subset_std0,
            "slope_minus_std": (
                slope - subset_std0 if pd.notna(slope) else np.nan
            ),
            "resid_mae": float(np.mean(np.abs(resid))),
            "resid_max_abs": float(np.max(np.abs(resid))),
        })

    return pd.DataFrame(rows)


# =============================================================================
# 1. 로드
# =============================================================================

banner("CN7 02 - 1425 -> 1211 원본행 복원")

source_path = find_file(BASE_DIR, SOURCE_NAMES)
target_path = find_file(BASE_DIR, TARGET_NAMES)

if source_path is None:
    raise FileNotFoundError(
        f"moldset_labeled.csv를 찾지 못했습니다.\nBASE_DIR={BASE_DIR}"
    )

if target_path is None:
    raise FileNotFoundError(
        f"moldset_labeled_cn7.csv를 찾지 못했습니다.\nBASE_DIR={BASE_DIR}"
    )

source_full = read_csv_flexible(source_path)
target = read_csv_flexible(target_path)

print("source:", source_path)
print("target:", target_path)
print("source full:", source_full.shape)
print("target     :", target.shape)

mask_cn7 = get_cn7_mask(source_full)
source = source_full.loc[mask_cn7].copy()

print("source CN7 :", source.shape)

if len(target) >= len(source):
    raise RuntimeError(
        f"예상과 다른 행 수입니다. source CN7={len(source)}, target={len(target)}"
    )

N_TARGET = len(target)

# 원본 index 고정
source["__source_row_index__"] = source.index

# Side 생성
part_name_col = find_col(source, ["PART_NAME"])
if part_name_col is not None:
    source["__Side__"] = extract_side(source[part_name_col])
else:
    source["__Side__"] = "UNKNOWN"

# =============================================================================
# 2. 공통 센서
# =============================================================================

banner("1) 공통 공정 변수")

pairs = common_numeric_features(source, target)

source_features = [x[0] for x in pairs]
target_features = [x[1] for x in pairs]

print("공통 numeric feature:", len(pairs))
for a, b in pairs:
    print(f"  {a} <-> {b}")

if len(pairs) < 5:
    raise RuntimeError("공통 numeric feature가 너무 적습니다.")

# =============================================================================
# 3. target에서 상수인 컬럼 -> 1425->1211 필터 단서
# =============================================================================

banner("2) Target 상수 컬럼으로 필터 규칙 탐색")

constant_features = []
constant_rule_candidates = []

for cs, ct in pairs:
    t_unique = target[ct].nunique(dropna=False)
    t_std = pd.to_numeric(target[ct], errors="coerce").std(ddof=0)

    if t_unique <= 1 or (pd.notna(t_std) and abs(t_std) < 1e-12):
        constant_features.append((cs, ct))

        vc = source[cs].value_counts(dropna=False)

        print(f"\nTarget constant: {ct}")
        print(f"Source {cs} value counts:")
        print(vc.head(20).to_string())

        for val, count in vc.items():
            if int(count) == N_TARGET:
                idx = source.index[source[cs].eq(val)].to_numpy()
                constant_rule_candidates.append({
                    "rule_name": f"{cs} == {val}",
                    "index": idx,
                    "source_feature": cs,
                    "value": val,
                })
                print(
                    f">>> EXACT {N_TARGET}개 규칙 발견: "
                    f"{cs} == {val}"
                )

print("\nTarget constant features:", [x[0] for x in constant_features])

# 여러 상수 컬럼의 값 조합도 탐색
if len(constant_features) >= 2:
    ccols = [x[0] for x in constant_features]

    combo_counts = (
        source.groupby(ccols, dropna=False)
        .size()
        .sort_values(ascending=False)
    )

    combo_df = combo_counts.reset_index(name="count")
    combo_df.to_csv(
        OUT_DIR / "constant_feature_combinations.csv",
        index=False,
        encoding="utf-8-sig",
    )

    for _, r in combo_df.iterrows():
        if int(r["count"]) == N_TARGET:
            m = pd.Series(True, index=source.index)
            desc = []

            for c in ccols:
                v = r[c]

                if pd.isna(v):
                    m &= source[c].isna()
                    desc.append(f"{c}=NaN")
                else:
                    m &= source[c].eq(v)
                    desc.append(f"{c}={v}")

            constant_rule_candidates.append({
                "rule_name": " & ".join(desc),
                "index": source.index[m].to_numpy(),
                "source_feature": "|".join(ccols),
                "value": "combination",
            })

# =============================================================================
# 4. metadata 단일 조건으로 1211개가 되는 규칙 탐색
# =============================================================================

banner("3) Metadata 단일/조합 필터 규칙 탐색")

metadata_cols = []

for c in source.columns:
    if c in source_features:
        continue
    if c in ["__source_row_index__"]:
        continue

    # 너무 고유한 자유텍스트/ID는 단일값 필터에서 제외하지 않되 출력만 제한
    metadata_cols.append(c)

metadata_exact_candidates = []

for c in metadata_cols:
    try:
        vc = source[c].value_counts(dropna=False)

        exact_vals = vc[vc == N_TARGET]

        if len(exact_vals):
            print(f"\n[{c}] 정확히 {N_TARGET}행 만드는 값:")
            print(exact_vals.to_string())

            for val, count in exact_vals.items():
                if pd.isna(val):
                    idx = source.index[source[c].isna()].to_numpy()
                    rule = f"{c} is NaN"
                else:
                    idx = source.index[source[c].eq(val)].to_numpy()
                    rule = f"{c} == {val}"

                metadata_exact_candidates.append({
                    "rule_name": rule,
                    "index": idx,
                    "source_feature": c,
                    "value": val,
                })
    except Exception:
        pass

# 날짜/시리얼/파트별 카운트 저장
meta_count_tables = []

for c in metadata_cols:
    nuniq = source[c].nunique(dropna=False)

    if nuniq <= 100:
        tmp = (
            source[c]
            .value_counts(dropna=False)
            .rename_axis("value")
            .reset_index(name="count")
        )
        tmp.insert(0, "column", c)
        meta_count_tables.append(tmp)

if meta_count_tables:
    pd.concat(meta_count_tables, ignore_index=True).to_csv(
        OUT_DIR / "metadata_value_counts.csv",
        index=False,
        encoding="utf-8-sig",
    )

# 후보 합치기 / 중복 rule 제거
all_rule_candidates = constant_rule_candidates + metadata_exact_candidates

dedup = {}
for r in all_rule_candidates:
    key = tuple(sorted(map(int, r["index"])))
    if len(key) == N_TARGET:
        dedup.setdefault(key, r)

all_rule_candidates = list(dedup.values())

print(f"\n정확히 {N_TARGET}행을 만드는 단순 규칙 후보: {len(all_rule_candidates)}개")

for i, r in enumerate(all_rule_candidates, 1):
    print(f"{i:2d}. {r['rule_name']}")

# =============================================================================
# 5. exact 1211 subset 후보들을 실제 z-score 재현으로 검증
# =============================================================================

banner("4) 후보 subset의 StandardScaler 재현 검증")

candidate_score_frames = []
candidate_outputs = {}

for i, r in enumerate(all_rule_candidates, 1):
    subset = source.loc[r["index"]].copy()

    if len(subset) != N_TARGET:
        continue

    name = f"rule_{i:02d}__{r['rule_name']}"

    scores, outputs = score_exact_subset(
        subset,
        target,
        source_features,
        target_features,
        subset_name=name,
    )

    candidate_score_frames.append(scores)
    candidate_outputs[name] = {
        "subset": subset,
        "outputs": outputs,
        "rule": r,
    }

    print(f"\n[{name}]")
    print(
        scores[
            [
                "mapping",
                "median_dist",
                "p95_dist",
                "max_dist",
                "exact_rate",
                "target_same_rate",
                "target_inverted_rate",
            ]
        ]
        .round(8)
        .to_string(index=False)
    )

if candidate_score_frames:
    candidate_scores = pd.concat(candidate_score_frames, ignore_index=True)

    candidate_scores = candidate_scores.sort_values(
        ["median_dist", "p95_dist", "max_dist"]
    )

    candidate_scores.to_csv(
        OUT_DIR / "exact_subset_candidate_scores.csv",
        index=False,
        encoding="utf-8-sig",
    )
else:
    candidate_scores = pd.DataFrame()
    print("정확히 1211행을 만드는 단순 규칙 후보가 없습니다.")

# =============================================================================
# 6. simple exact rule이 약하면 iterative Hungarian fallback
# =============================================================================

banner("5) Iterative assignment fallback")

iter_results = []
iter_outputs = {}

# exact candidate가 아주 좋으면 fallback도 참고용으로는 돌림.
# source 1425 x target 1211 정도라 계산 가능.
for mode in ["none", "same", "inverted"]:
    history, mapping, subset, mu, sd = iterative_assignment(
        source_all=source,
        target=target,
        source_feature_cols=source_features,
        target_feature_cols=target_features,
        label_mode=mode,
        max_iter=MAX_ITER,
    )

    history.to_csv(
        OUT_DIR / f"iterative_history__{mode}.csv",
        index=False,
        encoding="utf-8-sig",
    )

    final = history.iloc[-1].to_dict()
    final["method"] = f"iterative_{mode}"
    iter_results.append(final)

    iter_outputs[mode] = {
        "history": history,
        "mapping": mapping,
        "subset": subset,
        "mu": mu,
        "sd": sd,
    }

    print(f"\n[{mode}]")
    print(history.round(8).to_string(index=False))

iter_df = pd.DataFrame(iter_results)
iter_df.to_csv(
    OUT_DIR / "iterative_assignment_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 7. 최종 후보 선택
# =============================================================================

banner("6) 최종 복원 후보 선택")

final_kind = None
final_name = None
final_subset = None
final_mapping = None
final_mu = None
final_sd = None
final_metrics = None

# 1순위: exact rule + Hungarian 중 최저 거리
if len(candidate_scores):
    h = candidate_scores[
        candidate_scores["mapping"].str.startswith("hungarian")
    ].copy()

    if len(h):
        best = h.sort_values(
            ["median_dist", "p95_dist", "max_dist"]
        ).iloc[0]

        final_kind = "exact_rule"
        final_name = best["subset"]
        final_metrics = best.to_dict()

        obj = candidate_outputs[final_name]
        final_subset = obj["subset"]

        mapping_key = best["mapping"]
        final_mapping, final_mu, final_sd = obj["outputs"][mapping_key]

# 2순위: iterative
if final_mapping is None:
    best_iter = iter_df.sort_values(
        ["median_dist", "p95_dist", "max_dist"]
    ).iloc[0]

    mode = str(best_iter["label_mode"])

    final_kind = "iterative"
    final_name = f"iterative_{mode}"
    final_metrics = best_iter.to_dict()

    obj = iter_outputs[mode]
    final_subset = obj["subset"]
    final_mapping = obj["mapping"]
    final_mu = obj["mu"]
    final_sd = obj["sd"]

print("선택 method:", final_kind)
print("선택 name  :", final_name)
print("metrics    :", final_metrics)

# =============================================================================
# 8. 최종 매핑을 target 행 순서로 정렬 + 원본 metadata 붙이기
# =============================================================================

banner("7) 원본 metadata 복원")

final_mapping = final_mapping.sort_values("target_pos").reset_index(drop=True)

# target 원본 순서
target_ordered = target.loc[
    final_mapping["target_row_index"]
].copy().reset_index(drop=False)
target_ordered = target_ordered.rename(
    columns={"index": "__target_original_index__"}
)

source_ordered = source.loc[
    final_mapping["source_row_index"]
].copy().reset_index(drop=False)
source_ordered = source_ordered.rename(
    columns={"index": "__source_original_index__"}
)

# 중복 이름 충돌 방지를 위해 metadata 위주만 가져오기
metadata_restore_cols = []

for c in source_ordered.columns:
    if c in source_features:
        continue
    if c == target_col(source_ordered):
        continue
    metadata_restore_cols.append(c)

restored = target_ordered.copy()

for c in metadata_restore_cols:
    out_c = c
    if out_c in restored.columns:
        out_c = f"RAW__{c}"
    restored[out_c] = source_ordered[c].to_numpy()

# source target도 별도 보존
src_target_col = target_col(source_ordered)
tgt_target_col = target_col(target_ordered)

if src_target_col is not None:
    restored["__source_PassOrFail__"] = source_ordered[src_target_col].to_numpy()

if "__Side__" in source_ordered.columns:
    restored["Side"] = source_ordered["__Side__"].to_numpy()

# mapping distance
restored["__sensor_match_distance__"] = final_mapping[
    "sensor_distance"
].to_numpy()

restored.to_csv(
    OUT_DIR / "cn7_1211_restored.csv",
    index=False,
    encoding="utf-8-sig",
)

final_mapping.to_csv(
    OUT_DIR / "cn7_1211_row_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

print("복원 데이터 shape:", restored.shape)
print("sensor distance:")
print(restored["__sensor_match_distance__"].describe().to_string())

# =============================================================================
# 9. scaler 정확성 검증
# =============================================================================

banner("8) StandardScaler 정확성 검증")

# mapping에 포함된 source row 전체가 final subset이어야 함
selected_source_indices = final_mapping["source_row_index"].to_numpy()
selected_source = source.loc[selected_source_indices].copy()

# 중복 없이 target size인지 확인
print("선택 source 행 수:", len(selected_source_indices))
print("선택 source unique index:", len(np.unique(selected_source_indices)))

# subset 전체에서 scaler 계산
scaler_subset = source.loc[
    np.unique(selected_source_indices)
].copy()

affine = affine_column_check(
    subset=scaler_subset,
    target=target,
    mapping=final_mapping,
    source_feature_cols=source_features,
    target_feature_cols=target_features,
)

affine.to_csv(
    OUT_DIR / "scaler_reconstruction_check.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    affine[
        [
            "feature",
            "intercept_minus_mean",
            "slope_minus_std",
            "resid_mae",
            "resid_max_abs",
        ]
    ]
    .round(10)
    .to_string(index=False)
)

# scaler 파라미터 저장
_, actual_mu, actual_sd = scale_subset(
    scaler_subset,
    source_features,
)

scaler_params = pd.DataFrame({
    "feature": source_features,
    "mean_raw": [actual_mu[c] for c in source_features],
    "std0_raw": [actual_sd[c] for c in source_features],
})

scaler_params.to_csv(
    OUT_DIR / "recovered_scaler_parameters.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 10. Side / target / 시간 진단
# =============================================================================

banner("9) 복원 metadata 진단")

# PART_NAME / Side
if "Side" in restored.columns:
    print("\nSide counts:")
    print(restored["Side"].value_counts(dropna=False).to_string())

    if tgt_target_col is not None:
        print("\nTarget x Side:")
        side_ct = pd.crosstab(
            restored["Side"],
            restored[tgt_target_col],
            margins=True,
        )
        print(side_ct.to_string())
        side_ct.to_csv(
            OUT_DIR / "target_by_side.csv",
            encoding="utf-8-sig",
        )

# source target vs target target
if tgt_target_col is not None and "__source_PassOrFail__" in restored.columns:
    a = pd.to_numeric(
        restored["__source_PassOrFail__"],
        errors="coerce",
    )
    b = pd.to_numeric(
        restored[tgt_target_col],
        errors="coerce",
    )

    valid = a.notna() & b.notna()

    if valid.any():
        same = (a[valid].to_numpy() == b[valid].to_numpy()).mean()
        inv = ((1 - a[valid].to_numpy()) == b[valid].to_numpy()).mean()

        print("\nSource/target PassOrFail:")
        print("same rate    :", same)
        print("inverted rate:", inv)

# 시간
time_col = find_col(restored, ["TimeStamp", "RAW__TimeStamp"])

if time_col is not None:
    dt = pd.to_datetime(restored[time_col], errors="coerce")

    print("\nTimeStamp:")
    print("min:", dt.min())
    print("max:", dt.max())
    print("parsed:", dt.notna().sum(), "/", len(dt))

# serial
serial_col = find_col(
    restored,
    ["PART_FACT_SERIAL", "RAW__PART_FACT_SERIAL"],
)

if serial_col is not None:
    print("\nPART_FACT_SERIAL:")
    print(restored[serial_col].value_counts(dropna=False).to_string())

# plan date
plan_col = find_col(
    restored,
    ["PART_FACT_PLAN_DATE", "RAW__PART_FACT_PLAN_DATE"],
)

if plan_col is not None:
    print("\nPART_FACT_PLAN_DATE:")
    print(restored[plan_col].value_counts(dropna=False).to_string())

# =============================================================================
# 11. 제외된 214행과 선택된 1211행 비교
# =============================================================================

banner("10) 선택 1211 vs 제외 214 비교")

selected_idx_set = set(
    map(int, final_mapping["source_row_index"].tolist())
)

source_compare = source.copy()
source_compare["__selected_1211__"] = source_compare.index.map(
    lambda x: int(x) in selected_idx_set
)

print(source_compare["__selected_1211__"].value_counts().to_string())

# metadata별 선택률
filter_diag_rows = []

for c in source.columns:
    if c in source_features:
        continue
    if c.startswith("__"):
        continue

    try:
        g = (
            source_compare
            .groupby(c, dropna=False)["__selected_1211__"]
            .agg(["count", "sum", "mean"])
            .reset_index()
        )
        g["column"] = c
        g = g.rename(
            columns={
                "sum": "selected_count",
                "mean": "selected_rate",
            }
        )

        if len(g) <= 200:
            filter_diag_rows.append(
                g[
                    [
                        "column",
                        c,
                        "count",
                        "selected_count",
                        "selected_rate",
                    ]
                ].rename(columns={c: "value"})
            )
    except Exception:
        pass

if filter_diag_rows:
    filter_diag = pd.concat(
        filter_diag_rows,
        ignore_index=True,
    )
    filter_diag.to_csv(
        OUT_DIR / "selected_vs_excluded_metadata.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # 0% 또는 100% 선택되는 범주부터 출력
    strong = filter_diag[
        (filter_diag["selected_rate"] <= 0.001)
        | (filter_diag["selected_rate"] >= 0.999)
    ].copy()

    print("\n선택/제외를 완전히 가르는 metadata 값:")
    if len(strong):
        print(
            strong.sort_values(
                ["column", "selected_rate", "count"]
            )
            .to_string(index=False)
        )
    else:
        print("없음")

# 센서별 selected/excluded raw 분포
sensor_diag = []

for c in source_features:
    a = source_compare.loc[
        source_compare["__selected_1211__"], c
    ].astype(float)

    b = source_compare.loc[
        ~source_compare["__selected_1211__"], c
    ].astype(float)

    sensor_diag.append({
        "feature": c,
        "selected_n": len(a),
        "excluded_n": len(b),
        "selected_unique": a.nunique(dropna=False),
        "excluded_unique": b.nunique(dropna=False),
        "selected_mean": a.mean(),
        "excluded_mean": b.mean(),
        "selected_min": a.min(),
        "selected_max": a.max(),
        "excluded_min": b.min(),
        "excluded_max": b.max(),
    })

pd.DataFrame(sensor_diag).to_csv(
    OUT_DIR / "selected_vs_excluded_sensors.csv",
    index=False,
    encoding="utf-8-sig",
)

# =============================================================================
# 12. 자동 요약
# =============================================================================

banner("11) 자동 복원 요약")

summary = []

summary.append(
    f"- source CN7: {len(source)}행 -> target CN7: {len(target)}행"
)
summary.append(
    f"- 최종 복원 방법: {final_kind} / {final_name}"
)
summary.append(
    f"- sensor distance median: "
    f"{restored['__sensor_match_distance__'].median():.10g}"
)
summary.append(
    f"- sensor distance p95: "
    f"{restored['__sensor_match_distance__'].quantile(.95):.10g}"
)
summary.append(
    f"- exact(<= {EXACT_DISTANCE}) rate: "
    f"{(restored['__sensor_match_distance__'] <= EXACT_DISTANCE).mean():.3%}"
)

if "Side" in restored.columns:
    vc = restored["Side"].value_counts().to_dict()
    summary.append(
        f"- Side counts: {vc}"
    )

if tgt_target_col is not None and "__source_PassOrFail__" in restored.columns:
    a = pd.to_numeric(
        restored["__source_PassOrFail__"],
        errors="coerce",
    )
    b = pd.to_numeric(
        restored[tgt_target_col],
        errors="coerce",
    )
    valid = a.notna() & b.notna()

    if valid.any():
        same = (
            a[valid].to_numpy() == b[valid].to_numpy()
        ).mean()
        inv = (
            (1 - a[valid].to_numpy()) == b[valid].to_numpy()
        ).mean()

        summary.append(
            f"- PassOrFail same rate: {same:.6f}"
        )
        summary.append(
            f"- PassOrFail inverted rate: {inv:.6f}"
        )

if time_col is not None:
    dt = pd.to_datetime(
        restored[time_col],
        errors="coerce",
    )
    summary.append(
        f"- Time range: {dt.min()} ~ {dt.max()}"
    )

for x in summary:
    print(x)

with open(
    OUT_DIR / "restore_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(summary))

# =============================================================================
# 13. 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR.resolve())
print()
print("핵심 결과 파일:")
print("  1) exact_subset_candidate_scores.csv")
print("  2) iterative_assignment_summary.csv")
print("  3) cn7_1211_row_mapping.csv")
print("  4) cn7_1211_restored.csv")
print("  5) scaler_reconstruction_check.csv")
print("  6) recovered_scaler_parameters.csv")
print("  7) selected_vs_excluded_metadata.csv")
print("  8) selected_vs_excluded_sensors.csv")
print("  9) target_by_side.csv")
print(" 10) restore_summary.txt")
print()
print(
    "다음 단계에서는 restore_summary.txt와 "
    "cn7_1211_restored.csv를 기준으로 EDA를 시작합니다."
)



CN7 02 - 1425 -> 1211 원본행 복원
source: C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled.csv
target: C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv
source full: (2607, 47)
target     : (1211, 26)
source CN7 : (1425, 47)

1) 공통 공정 변수
공통 numeric feature: 24
  Injection_Time <-> Injection_Time
  Filling_Time <-> Filling_Time
  Plasticizing_Time <-> Plasticizing_Time
  Cycle_Time <-> Cycle_Time
  Clamp_Close_Time <-> Clamp_Close_Time
  Cushion_Position <-> Cushion_Position
  Plasticizing_Position <-> Plasticizing_Position
  Clamp_Open_Position <-> Clamp_Open_Position
  Max_Injection_Speed <-> Max_Injection_Speed
  Max_Screw_RPM <-> Max_Screw_RPM
  Average_Screw_RPM <-> Average_Screw_RPM
  Max_Injection_Pressure <-> Max_Injection_Pressure
  Max_Switch_Over_Pressure <-> Max_Switch_Over_Pressure
  Max_Back_Pressure <-> Max_Back_Pressure
  Average_Back_Pressure <-> Average_Back_Pressure
  Barrel_Temperature_1 <->